# 30_pose_bakeoff

Body-language branch, part 1: **dog keypoints from video**, then a small **pose decoder**
that turns a keypoint sequence into 8 body-language classes, folded into the 5 shared
`woofdecoder.taxonomy.CATEGORIES` for fusion.

Owners: Rohan M., Jackson O.

| Step | What | Decides (`configs/pipeline.yaml`) |
|---|---|---|
| 1 | Setup | |
| 2 | Zero-shot check of the COCO-pretrained YOLO11-pose | |
| 3 | Export StanfordExtra (team splits) to YOLO pose format | |
| 4 | Fine-tune YOLO11-pose on dog keypoints, compare sizes | `branches.body.pose` |
| 5 | Run the fine-tuned model on a video | `branches.body.detector` |
| 6 | Keypoints → pose features | |
| 7 | 8 body classes, mapping to the 5 shared categories, label sources | |
| 8 | Pose decoder (architecture + shape check) | `branches.body.head` |

Pretrained model: [Ultralytics YOLO11-pose](https://docs.ultralytics.com/tasks/pose/)
(`yolo11n-pose.pt`, COCO human pose, 17 keypoints). We keep its backbone and retrain the
keypoint head for the 24 dog keypoints, following the Ultralytics Dog-Pose recipe
([docs](https://docs.ultralytics.com/datasets/pose/dog-pose/),
[video](https://www.youtube.com/watch?v=ZhjO32tZUek)).

**Difference from the video:** the video trains on `dog-pose.yaml` directly. Its `val` folder is
StanfordExtra's **test** split (see `src/woofdata/prepare/sources/dog_pose.py`), so validating
on it leaks the test set. Here we build the YOLO dataset from the team's `stanfordextra`
train / val splits instead, and report test once at the end.

## 1. Setup

`ultralytics` is not in `pyproject.toml` yet. Install it into the repo `.venv` once
(it also pulls in `torch` and `opencv-python`). Add it to the `ml` extra when this
notebook settles the model.

In [ ]:
%pip install -q ultralytics

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from ultralytics import YOLO

from woofdata.loader import load_sample
from woofdata.path import data_path, get_data_dir, load_data_dir_from_env_file
from woofdata.prepare.sources.stanfordextra import EMPTY_KEYPOINTS, KEYPOINT_NAMES
from woofdata.tasks import task_rows
from woofdecoder.taxonomy import CATEGORIES, NUM_CATEGORIES

load_data_dir_from_env_file()
DATA_DIR = get_data_dir()
REPO_DIR = Path.cwd().parent  # notebooks/ -> repo root
MODELS_DIR = REPO_DIR / "models" / "pose"  # gitignored
YOLO_DATA_DIR = DATA_DIR / "processed" / "yolo_dog_pose"  # built in step 3

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

NUM_KEYPOINTS = len(KEYPOINT_NAMES)  # 24
print(
    f"data dir: {DATA_DIR}\ndevice: {DEVICE}\nkeypoints: {NUM_KEYPOINTS}, categories: {CATEGORIES}"
)

## 2. Zero-shot check

The pretrained `yolo11n-pose.pt` only knows one class (`person`) and 17 human keypoints, so on
a dog image it should find nothing (or a wrong skeleton). This is why the keypoint head has to be
retrained.

In [ ]:
pose_rows = {split: task_rows("pose", split) for split in ("train", "val")}
for split, rows in pose_rows.items():
    print(f"{split}: {len(rows)} images")

coco_pose_model = YOLO("yolo11n-pose.pt")  # downloads ~6 MB on first use
sample = load_sample("pose", pose_rows["val"].iloc[0])
zero_shot = coco_pose_model.predict(sample.image, device=DEVICE, verbose=False)[0]
print(f"classes the pretrained model knows: {coco_pose_model.names}")
print(f"detections on a dog image: {len(zero_shot.boxes)}")
plt.imshow(zero_shot.plot()[:, :, ::-1])
plt.axis("off");

## 3. Export StanfordExtra to YOLO pose format

One `.txt` label per image: `class cx cy w h` then 24 × `x y v`, all normalised to the image
size (the same format as Dog-Pose). Images are symlinked, not copied.

Keypoints [20, 21, 22, 23] (eyes, withers, throat) are never labelled in StanfordExtra
(`EMPTY_KEYPOINTS`), so the model cannot learn them. We keep all 24 so the label format matches
Dog-Pose, and drop them later in step 6.

`flip_idx` tells YOLO which keypoint to swap with which when an image is flipped left-right
during augmentation.

In [ ]:
# Left-right partner of each keypoint (KEYPOINT_NAMES order).
FLIP_IDX = [6, 7, 8, 9, 10, 11, 0, 1, 2, 3, 4, 5, 12, 13, 15, 14, 16, 17, 19, 18, 21, 20, 22, 23]
for position, partner in enumerate(FLIP_IDX):
    name, partner_name = KEYPOINT_NAMES[position], KEYPOINT_NAMES[partner]
    assert (
        name.replace("left", "right") == partner_name
        or name.replace("right", "left") == partner_name
    ), (
        name,
        partner_name,
    )


def yolo_label_line(box: np.ndarray, keypoints: np.ndarray, width: int, height: int) -> str:
    """Return one YOLO pose label line for a dog box (x, y, w, h) and its (24, 3) keypoints."""
    x, y, w, h = box
    values = [0, (x + w / 2) / width, (y + h / 2) / height, w / width, h / height]
    for kx, ky, visibility in keypoints:
        if visibility > 0:
            values += [kx / width, ky / height, 2]
        else:
            values += [0.0, 0.0, 0]
    return " ".join(f"{value:.6g}" for value in values)


def export_split(split: str) -> int:
    """Write images/<split>/ symlinks and labels/<split>/ files; return the image count."""
    image_dir = YOLO_DATA_DIR / "images" / split
    label_dir = YOLO_DATA_DIR / "labels" / split
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    rows = pose_rows[split]
    for _, row in rows.iterrows():
        sample = load_sample("pose", row)
        height, width = sample.image.shape[:2]
        source = data_path(row["file_path"])
        stem = row["clip_id"].replace("/", "__")  # clip ids look like source/key
        link = image_dir / f"{stem}{source.suffix}"
        if not link.exists():
            link.symlink_to(source)
        line = yolo_label_line(sample.box, sample.keypoints, width, height)
        (label_dir / f"{stem}.txt").write_text(line + "\n")
    return len(rows)


for split in ("train", "val"):
    print(f"{split}: {export_split(split)} images written")

DATASET_YAML = YOLO_DATA_DIR / "dog_pose_team.yaml"
DATASET_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(YOLO_DATA_DIR),
            "train": "images/train",
            "val": "images/val",
            "kpt_shape": [NUM_KEYPOINTS, 3],
            "flip_idx": FLIP_IDX,
            "names": {0: "dog"},
            "kpt_names": {0: list(KEYPOINT_NAMES)},
        },
        sort_keys=False,
    )
)
print(DATASET_YAML.read_text())

## 4. Fine-tune YOLO11-pose

Same call as the video (`yolo pose train data=... model=yolo11n-pose.pt epochs=50 imgsz=640`),
in Python. Ultralytics swaps the 17-keypoint head for a 24-keypoint one and keeps the
pretrained backbone.

The video stopped at 20 epochs and the pose mAP was still climbing, so 50 is a starting
point, not a decision. On a laptop CPU/MPS one epoch of `n` is slow: run the real training on
a GPU (Colab L4, as in the video, or the CSSE cluster) with `SMOKE_TEST = False`.

**Bakeoff:** train each size in `CANDIDATES` and compare pose mAP50-95 on val against
inference time per frame. The app runs ~4 frames per bark (`media.frames_per_segment`), so
a bigger model is affordable if it is clearly more accurate.

In [ ]:
SMOKE_TEST = True  # True: 1 epoch on 5% of the data to check the pipeline runs end to end

CANDIDATES = ["yolo11n-pose.pt", "yolo11s-pose.pt"]  # add "yolo11m-pose.pt" if s beats n clearly
EPOCHS = 1 if SMOKE_TEST else 50
IMAGE_SIZE = 640
BATCH = 16
PATIENCE = 10  # stop early when val pose mAP stops improving

bakeoff = {}
for weights in CANDIDATES:
    run_name = Path(weights).stem + ("_smoke" if SMOKE_TEST else "")
    model = YOLO(weights)
    model.train(
        data=str(DATASET_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        patience=PATIENCE,
        fraction=0.05 if SMOKE_TEST else 1.0,
        device=DEVICE,
        project=str(MODELS_DIR),
        name=run_name,
        exist_ok=True,
        seed=0,
    )
    metrics = model.val(
        data=str(DATASET_YAML),
        split="val",
        device=DEVICE,
        project=str(MODELS_DIR),
        name=f"{run_name}_val",
        exist_ok=True,
    )
    bakeoff[weights] = {
        "best_weights": MODELS_DIR / run_name / "weights" / "best.pt",
        "pose_mAP50-95": metrics.pose.map,
        "pose_mAP50": metrics.pose.map50,
        "box_mAP50-95": metrics.box.map,
        "ms_per_image": metrics.speed["inference"],
    }

for weights, result in bakeoff.items():
    print(weights, {key: value for key, value in result.items() if key != "best_weights"})

Training curves and the val batch predictions (labels vs predictions, as shown in the video)
are saved in `models/pose/<run>/` (`results.png`, `val_batch*_pred.jpg`).

In [ ]:
BEST_WEIGHTS = bakeoff[CANDIDATES[0]]["best_weights"]  # change after comparing the bakeoff
pose_model = YOLO(BEST_WEIGHTS)

figure, axes = plt.subplots(1, 4, figsize=(16, 4))
for axis, (_, row) in zip(axes, pose_rows["val"].sample(4, random_state=0).iterrows()):
    result = pose_model.predict(load_sample("pose", row).image, device=DEVICE, verbose=False)[0]
    axis.imshow(result.plot()[:, :, ::-1])
    axis.axis("off")

## 5. Keypoints from a video

The app's input is an uploaded video. For now this samples frames at a fixed rate; in the
pipeline the frames come from each `BarkSegment` (`segment.frames`, `segment.dog_boxes`).

If several dogs are detected, keep the most confident one. In the pipeline the dog to keep is
the one matching `segment.dog_boxes`.

In [ ]:
import cv2

VIDEO_PATH = None  # e.g. DATA_DIR / "raw" / "my_clips" / "dog.mp4"
SAMPLE_FPS = 4.0


def read_frames(video_path: Path, sample_fps: float) -> tuple[list[np.ndarray], list[float]]:
    """Return RGB frames sampled at sample_fps and their timestamps in seconds."""
    capture = cv2.VideoCapture(str(video_path))
    video_fps = capture.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(1, round(video_fps / sample_fps))
    frames, times = [], []
    index = 0
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        if index % step == 0:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            times.append(index / video_fps)
        index += 1
    capture.release()
    return frames, times


def dog_keypoints(model: YOLO, frames: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """Return keypoints, shape (T, 24, 3) as x, y, confidence in pixels, and boxes, shape
    (T, 4) as x, y, w, h, for the most confident dog in each frame (NaN if none)."""
    keypoints = np.full((len(frames), NUM_KEYPOINTS, 3), np.nan, dtype=np.float32)
    boxes = np.full((len(frames), 4), np.nan, dtype=np.float32)
    for position, result in enumerate(model.predict(frames, device=DEVICE, verbose=False)):
        if len(result.boxes) == 0:
            continue
        best = int(result.boxes.conf.argmax())
        keypoints[position] = result.keypoints.data[best].cpu().numpy()
        x1, y1, x2, y2 = result.boxes.xyxy[best].cpu().numpy()
        boxes[position] = [x1, y1, x2 - x1, y2 - y1]
    return keypoints, boxes


if VIDEO_PATH is not None:
    video_frames, video_times = read_frames(VIDEO_PATH, SAMPLE_FPS)
    video_keypoints, video_boxes = dog_keypoints(pose_model, video_frames)
    found = ~np.isnan(video_boxes[:, 0])
    print(f"{len(video_frames)} frames, dog found in {found.sum()}")

## 6. Keypoints → pose features

Raw pixel keypoints depend on where the dog is and how big it is. Normalise each frame to the
dog's box (centre → 0, box size → 1) and drop the 4 never-labelled keypoints, giving
20 × (x, y, confidence) = 60 numbers per frame.

A few hand-made cues, from what people read in dog body language (all relative to the box,
image y grows downward so the signs are flipped):

| Cue | From | Reads as |
|---|---|---|
| tail height | `tail_end` vs `tail_start` | tail up (alert/excited) vs tucked (fearful) |
| ear height | ear tips vs ear bases | ears up vs pinned back |
| head height | `nose` vs box centre | head up vs lowered |
| box aspect | box w / h | standing vs lying / play bow |

Whether the cues help on top of the 60 raw numbers is a question for notebook 31.
They also give a no-training baseline: hand rules from the chart (e.g. tail up + ears up → alert).

In [ ]:
KEPT_KEYPOINTS = [i for i in range(NUM_KEYPOINTS) if i not in EMPTY_KEYPOINTS]  # 20 kept
KP = {name: position for position, name in enumerate(KEYPOINT_NAMES)}
NUM_CUES = 4
FEATURES_PER_FRAME = len(KEPT_KEYPOINTS) * 3 + NUM_CUES  # 64


def box_y(frame_keypoints: np.ndarray, box: np.ndarray, name: str) -> float:
    """Height of one keypoint relative to the box: 0 at the centre, +0.5 at the bottom."""
    _, y, _, h = box
    return (frame_keypoints[KP[name], 1] - (y + h / 2)) / h


def pose_features(keypoints: np.ndarray, boxes: np.ndarray) -> np.ndarray:
    """Turn (T, 24, 3) pixel keypoints and (T, 4) boxes into (T, FEATURES_PER_FRAME)
    box-normalised features. Frames with no dog are all zeros."""
    num_frames = len(keypoints)
    features = np.zeros((num_frames, FEATURES_PER_FRAME), dtype=np.float32)
    for t in range(num_frames):
        if np.isnan(boxes[t, 0]):
            continue
        x, y, w, h = boxes[t]
        points = keypoints[t, KEPT_KEYPOINTS].copy()  # shape (20, 3)
        points[:, 0] = (points[:, 0] - (x + w / 2)) / w
        points[:, 1] = (points[:, 1] - (y + h / 2)) / h

        frame, box = keypoints[t], boxes[t]
        tail_height = box_y(frame, box, "tail_start") - box_y(frame, box, "tail_end")
        ear_height = (
            box_y(frame, box, "left_ear_base")
            + box_y(frame, box, "right_ear_base")
            - box_y(frame, box, "left_ear_tip")
            - box_y(frame, box, "right_ear_tip")
        ) / 2
        head_height = -box_y(frame, box, "nose")
        aspect = w / h
        features[t] = np.concatenate(
            [points.ravel(), [tail_height, ear_height, head_height, aspect]]
        )
    return features


if VIDEO_PATH is not None:
    video_features = pose_features(video_keypoints, video_boxes)
    print(video_features.shape)

## 7. Body classes and labels

The body branch predicts **8 pose classes**, finer than the 5 shared categories, because some
categories that sound alike look different (an alert dog stands tall and still; an aggressive
one leans forward with a stiff tail; a fearful one crouches with the tail tucked). Fusion still
gets 5 numbers: each shared category's probability is the sum of its pose classes.

| Pose class | Shared category | Looks like (cf. the "Body language of a dog" chart) |
|---|---|---|
| `alert` | alert_aggressive | Alert: tall, ears up and forward, tail up |
| `aggressive` | alert_aggressive | I need space / stiff, weight forward |
| `fearful_defensive` | alert_aggressive | Scared, Defensive: low, ears back, tail tucked |
| `playful_excited` | playful_excited | Let's play: play bow, loose body |
| `distressed` | distressed_lonely | pacing, lip licking, ears back, low tail |
| `lonely` | distressed_lonely | lying or sitting by a door, head low |
| `attention_seeking` | attention_seeking | Please: sitting, looking up at a person |
| `other` | other | Comfortable, De-stress, I like you: relaxed |

**Labels.** We never label what the dog *feels*. Image/video sets labelled by the visible
body-language class give us training labels for the decoder:

| Dataset | Labels | Size | Maps to |
|---|---|---|---|
| [Roboflow Dog Emotion Videos](https://universe.roboflow.com/dog-emotion-classification-rp9gv/dog-emotion-videos-akggu) | Aggressive, Anxiety, Fear, Happy | 1,873 frames from videos | aggressive, distressed, fearful_defensive, playful_excited |
| [Kaggle Dog Emotion](https://www.kaggle.com/dsv/4969612) | angry, happy, relaxed, sad | 4,000 images (1,000 each) | aggressive, playful_excited, other, distressed |

No public set has `alert`, `lonely` or `attention_seeking`: we label those from our own
YouTube clips by **situation** (doorbell → alert, owner leaves → lonely, dog at the food bowl
looking at a person → attention_seeking), as in the repo's labelling plan.

In [ ]:
BODY_CLASSES = [
    "alert",
    "aggressive",
    "fearful_defensive",
    "playful_excited",
    "distressed",
    "lonely",
    "attention_seeking",
    "other",
]
BODY_TO_CATEGORY = {
    "alert": "alert_aggressive",
    "aggressive": "alert_aggressive",
    "fearful_defensive": "alert_aggressive",
    "playful_excited": "playful_excited",
    "distressed": "distressed_lonely",
    "lonely": "distressed_lonely",
    "attention_seeking": "attention_seeking",
    "other": "other",
}
NUM_BODY_CLASSES = len(BODY_CLASSES)

# (NUM_BODY_CLASSES, NUM_CATEGORIES) 0/1 matrix: body probs @ matrix = shared category probs.
BODY_TO_CATEGORY_MATRIX = torch.zeros(NUM_BODY_CLASSES, NUM_CATEGORIES)
for position, body_class in enumerate(BODY_CLASSES):
    BODY_TO_CATEGORY_MATRIX[position, CATEGORIES.index(BODY_TO_CATEGORY[body_class])] = 1.0
assert (BODY_TO_CATEGORY_MATRIX.sum(1) == 1).all()


def category_probs(body_logits: torch.Tensor) -> torch.Tensor:
    """(B, NUM_BODY_CLASSES) logits -> (B, NUM_CATEGORIES) probabilities for fusion."""
    return body_logits.softmax(dim=-1) @ BODY_TO_CATEGORY_MATRIX

## 8. Pose decoder

Input: one bark segment's frames as features, shape (batch, T, FEATURES_PER_FRAME).
Output: logits over the 8 `BODY_CLASSES`; `category_probs` folds them into the 5 shared
categories for the branch's `BranchOutput.probs`.

```
(B, T, 64) ─► per-frame MLP ─► (B, T, H) ─► temporal pooling ─► (B, H) ─► MLP ─► (B, 8) ─► sum ─► (B, 5)
                                               mean  |  GRU
```

Per-frame MLP first so every frame is read the same way, then pooling over time so the
decoder works for any number of frames (a segment can lose frames where no dog is found).

**Hyperparameters still open** (pick on val in notebook 31 once we have labelled clips):

| Name | Start value | Options to try |
|---|---|---|
| `NUM_FRAMES` (T) | 4 (`media.frames_per_segment`) | 4, 8, 16 |
| `HIDDEN` | 128 | 64, 128, 256 |
| `TEMPORAL` | `"mean"` | `"mean"`, `"gru"` |
| `DROPOUT` | 0.3 | 0.1–0.5 |
| learning rate | 1e-3 (AdamW) | 3e-4, 1e-3 |
| weight decay | 1e-2 | |
| loss | cross-entropy, class weights from label counts | + label smoothing 0.1 |

In [ ]:
from torch import nn

NUM_FRAMES = 4
HIDDEN = 128
TEMPORAL = "mean"
DROPOUT = 0.3


class PoseDecoder(nn.Module):
    """Pose feature sequence (B, T, F) -> body-class logits (B, NUM_BODY_CLASSES)."""

    def __init__(
        self,
        in_features: int = FEATURES_PER_FRAME,
        hidden: int = HIDDEN,
        temporal: str = TEMPORAL,
        dropout: float = DROPOUT,
        num_classes: int = NUM_BODY_CLASSES,
    ) -> None:
        super().__init__()
        self.frame_encoder = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Linear(in_features, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        self.temporal = temporal
        if temporal == "gru":
            self.gru = nn.GRU(hidden, hidden, batch_first=True)
        elif temporal != "mean":
            raise ValueError(f"temporal must be 'mean' or 'gru', not {temporal!r}")
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, num_classes))

    def forward(
        self, features: torch.Tensor, frame_mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        """features: (B, T, F). frame_mask: (B, T) bool, False where no dog was found."""
        encoded = self.frame_encoder(features)  # (B, T, H)
        if self.temporal == "gru":
            _, last_hidden = self.gru(encoded)  # (1, B, H)
            pooled = last_hidden[0]
        else:
            if frame_mask is None:
                frame_mask = torch.ones(
                    features.shape[:2], dtype=torch.bool, device=features.device
                )
            weights = frame_mask.unsqueeze(-1).float()  # (B, T, 1)
            pooled = (encoded * weights).sum(1) / weights.sum(1).clamp(min=1.0)  # (B, H)
        return self.classifier(pooled)  # (B, NUM_BODY_CLASSES)


# Shape check with random input: the decoder is untrained until the labelled sets are prepared.
for temporal in ("mean", "gru"):
    decoder = PoseDecoder(temporal=temporal)
    logits = decoder(torch.randn(8, NUM_FRAMES, FEATURES_PER_FRAME))
    probs = category_probs(logits)
    num_params = sum(p.numel() for p in decoder.parameters())
    print(
        f"{temporal}: body logits {tuple(logits.shape)}, category probs {tuple(probs.shape)}, "
        f"sum {probs.sum(-1)[0]:.3f}, {num_params:,} params"
    )

## Open questions

- **Labelled sets.** Add the two sets from step 7 to `configs/datasets.yaml` with converters
  (frames → pose features → body class), and a `body_language` task in `woofdata.tasks`.
- **Image vs video labels.** Both public sets are single frames; train with T = 1 first, then
  move to T = 4 on our own video clips. The `posture` task (standing / sitting / lying) can
  pretrain the decoder (notebook 31).
- **Which model size** after the bakeoff (step 4), then write `branches.body.pose` and
  `branches.body.head` into `configs/pipeline.yaml` and move `dog_keypoints`, `pose_features`
  and `PoseDecoder` into `src/woofdecoder/branches/body.py`.